# 02.1 · Enrich papers data

In [1]:
import os
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv, find_dotenv

DATA = Path("../data")
CACHE = DATA / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
load_dotenv(dotenv_path=find_dotenv())                                   # tokens live in ../.env, never in the notebook

True

# OpenAlex Works Crawler

Crawl a set of DOIs through the OpenAlex API and produce two tidy parquet files: one row per work (`works.parquet`) and one row per field (`fields.parquet`).

## 📥 Input

| Path | Required column | Notes |
|---|---|---|
| `../data/cache/dois.parquet` | `doi` | Any format — URLs, `doi:` prefixes, mixed case are all normalized. |

## 📤 Output

### `../data/works.parquet`

| Column | Type | Description |
|---|---|---|
| `openalex_id` | `str` | Short OpenAlex ID, e.g. `W2741809807` |
| `doi` | `str` | Normalized DOI (lowercase, no `https://doi.org/` prefix) |
| `apc_paid_value_usd` | `float` | APC actually paid, in USD (`NaN` if unknown) |
| `apc_list_value_usd` | `float` | APC list price, in USD (`NaN` if unknown or diamond OA = `0.0`) |
| `awards` | `list[str]` | Funder award IDs linked to the work |
| `funders` | `list[str]` | Funder IDs (OpenAlex `F…`) |
| `primary_topic_field_id` | `str` | Field ID (`F…`) of the top-ranked topic |
| `topics_field_ids` | `list[str]` | Field IDs of all topics, in OpenAlex's ranking order |
| `referenced_works` | `list[str]` | Short IDs (`W…`) of cited works |

### `../data/fields.parquet`

Reference table — **the only place human-readable taxonomy labels live**.

| Column | Type | Description |
|---|---|---|
| `field_id` | `str` | e.g. `F1234` |
| `field_name` | `str` | e.g. `"Oncology"` |
| `domain_id` | `str` | e.g. `D0001` |
| `domain_name` | `str` | e.g. `"Health Sciences"` |

≈26 rows. Regenerated on every run (one API request).

## ⚙️ How it works

1. **Normalize** every DOI (`https://doi.org/…`, `doi:…`, uppercase → canonical lowercase form).
2. **Resume**: if `works.parquet` already exists, its DOIs are excluded from the input, so only new works are fetched.
3. **Batch**: DOIs are grouped in chunks of **50** and sent as a single `filter=doi:A|B|C…` request. `per_page=50` is set so a batch never spills to page 2.
4. **Parallelize**: up to **5 batches in flight** via a thread pool. I/O-bound → threads, not processes.
5. **Retry** with exponential backoff (`2^n` seconds, capped at 30 s, up to 5 attempts) on:
   - network errors
   - HTTP 5xx
   - HTTP 429 (rate-limited)
6. **Do not retry** on other HTTP 4xx (malformed DOI, etc.) — the batch is skipped with a warning and its DOIs are retried on the next run.
7. **Merge**: new rows are concatenated with the existing file, deduplicated on `openalex_id`, and rewritten.

## 🔧 Configuration

| Variable | Default | What it does |
|---|---|---|
| `OPENALEX_EMAIL` | `"your-email@example.com"` | Joins the polite pool. **Set this.** |
| `OPENALEX_API_KEY` | *required* | Read from environment; the script fails fast if unset. |
| `BATCH_SIZE` | `50` | DOIs per request. Max 100 per OR-filter. |
| `PER_PAGE` | `50` | Must be ≥ `BATCH_SIZE`. |
| `MAX_WORKERS` | `5` | Parallel batches. Lower if you hit 429s. |
| `MAX_RETRIES` | `5` | Retry attempts for transient errors. |

## 🔗 Joining the two files

Attach names to the primary topic:

```python
works  = pd.read_parquet(DATA / "works.parquet")
fields = pd.read_parquet(DATA / "fields.parquet")

works = works.merge(
    fields.rename(columns={
        "field_id":    "primary_topic_field_id",
        "field_name":  "primary_topic_field_name",
        "domain_id":   "primary_topic_domain_id",
        "domain_name": "primary_topic_domain_name",
    }),
    on="primary_topic_field_id", how="left",
)
```

Explode all topics (one row per work×topic):

```python
per_topic = (
    works[["openalex_id", "topics_field_ids"]]
    .explode("topics_field_ids")
    .merge(fields, left_on="topics_field_ids", right_on="field_id", how="left")
    .drop(columns="field_id")
)
```

Build the internal citation graph:

```python
known = set(works["openalex_id"])
internal_cites = (
    works.explode("referenced_works")
         .query("referenced_works in @known")
)
```

## ⚠️ Caveats

- **Append-only**: DOIs removed from the input are **not** removed from `works.parquet`. To rebuild, delete the file.
- **No refresh**: an already-fetched work is never re-queried. To refresh metadata, delete `works.parquet`.
- **Missing DOIs**: works not present in OpenAlex are silently omitted (no stub row). They cost one lookup on each run — keep an exclusion list if this becomes expensive.
- **No total funding**: OpenAlex doesn't expose a single aggregate amount; `awards` lists award IDs only. Funding amounts live on the award entity and are not fetched here.
- **`NaN` in APC columns** is a valid signal (closed access, unknown, or not applicable) — not a bug.
- **`primary_topic_field_id` is `None`** for works the classifier can't assign a topic to.
- **Schema drift**: adding a column to `REQUESTED` later leaves old rows with `NaN` there. The loader prints a warning so you can decide whether to re-fetch.

In [ ]:
import os
import time
import threading
from pathlib import Path

import pandas as pd
import requests
import pyalex
from pyalex import Works
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# --- Config -----------------------------------------------------------------
pyalex.config.email   = os.environ["OPENALEX_EMAIL"]
pyalex.config.api_key = os.environ["OPENALEX_API_KEY"]   # must be set in env

# Our own retry loop owns retries, so that a stop signal can interrupt backoff.
pyalex.config.max_retries      = 0
pyalex.config.retry_http_codes = [429, 500, 503]

FIELDS            = ["ids", "apc_list", "apc_paid", "awards",
                     "funders", "primary_topic", "topics", "referenced_works"]
BATCH_SIZE        = 50
PER_PAGE          = 50          # must be >= BATCH_SIZE
MAX_WORKERS       = 5
MAX_RETRIES       = 5
CHECKPOINT_EVERY  = 50           # batches (= 500 DOIs) per checkpoint

# A 429 asking us to wait longer than this is treated as "budget spent" and
# triggers a graceful stop instead of blocking the notebook for minutes.
LONG_WAIT_SECONDS = 120

CHECK_QUOTA_FIRST = True  # optional /rate-limit probe before crawling

DATA = Path("../data")

REQUESTED = [
    "openalex_id", "doi",
    "apc_paid_value_usd", "apc_list_value_usd",
    "awards", "funders",
    "primary_topic_field_id", "topics_field_ids",
    "referenced_works",
]
LIST_COLS = ["awards", "funders", "topics_field_ids", "referenced_works"]


class CrawlHalted(RuntimeError):
    """Hard stop: today's API budget is spent (or the crawl was aborted)."""


# --- Helpers ----------------------------------------------------------------
def normalize_doi(doi):
    if pd.isna(doi):
        return None
    doi = str(doi).strip().lower()
    for p in ("https://doi.org/", "http://doi.org/",
              "https://dx.doi.org/", "http://dx.doi.org/", "doi:"):
        if doi.startswith(p):
            doi = doi[len(p):]
    return doi.strip() or None


def short_id(url):
    """'https://openalex.org/W123' -> 'W123'; None-safe."""
    if not url:
        return None
    return url.rstrip("/").rsplit("/", 1)[-1]


def _quota_reason(exc):
    """Return a human reason if `exc` means today's budget is spent, else None."""
    resp = getattr(exc, "response", None)
    if getattr(resp, "status_code", None) != 429:
        return None

    h = getattr(resp, "headers", {}) or {}
    remaining   = h.get("X-RateLimit-Remaining")
    retry_after = h.get("Retry-After")

    try:
        if remaining is not None and int(float(remaining)) <= 0:
            return f"daily budget spent (X-RateLimit-Remaining={remaining})"
    except (TypeError, ValueError):
        pass
    try:
        if retry_after is not None and float(retry_after) > LONG_WAIT_SECONDS:
            return f"server asks for a {retry_after}s wait (Retry-After)"
    except (TypeError, ValueError):
        pass
    return None


def fetch_batch(dois, stop, attempt=0):
    """Fetch one batch. Raises CrawlHalted when the budget is exhausted."""
    if stop.is_set():
        return []                       # aborted before we started; not cached anywhere

    try:
        return (Works()
                .filter_or(doi=dois)
                .select(FIELDS)
                .get(per_page=PER_PAGE))

    except Exception as e:
        status = getattr(getattr(e, "response", None), "status_code", None)

        if status == 429:
            reason = _quota_reason(e)
            if reason:
                stop.set()
                raise CrawlHalted(reason) from e
            if attempt >= MAX_RETRIES:
                stop.set()
                raise CrawlHalted(f"persistent 429 after {MAX_RETRIES} retries") from e
            if stop.wait(min(2 ** attempt, 30)):        # interruptible backoff
                raise CrawlHalted("stop requested during backoff") from e
            return fetch_batch(dois, stop, attempt + 1)

        # Other 4xx: bad DOI / forbidden / not found — retrying won't help.
        if status is not None and 400 <= status < 500:
            print(f"  ! HTTP {status} on batch of {len(dois)} DOIs — skipping (no retry)")
            return []

        # 5xx and network errors: retry with backoff, then give up on this batch.
        if attempt >= MAX_RETRIES:
            raise
        if stop.wait(min(2 ** attempt, 30)):
            raise CrawlHalted("stop requested during backoff") from e
        return fetch_batch(dois, stop, attempt + 1)


def to_row(work):
    ids      = work.get("ids") or {}
    apc_paid = (work.get("apc_paid") or {}).get("value_usd")
    apc_list = (work.get("apc_list") or {}).get("value_usd")
    awards   = work.get("awards")           or []
    funders  = work.get("funders")          or []
    topics   = work.get("topics")           or []
    refs     = work.get("referenced_works") or []

    primary = work.get("primary_topic") or {}
    pfield  = primary.get("field") or {}

    return {
        "openalex_id":              short_id(ids.get("openalex")),
        "doi":                      normalize_doi(ids.get("doi")),
        "apc_paid_value_usd":       apc_paid,
        "apc_list_value_usd":       apc_list,
        "awards":                   [a.get("funder_award_id") for a in awards
                                     if a.get("funder_award_id")],
        "funders":                  [short_id(f.get("id")) for f in funders if f.get("id")],
        "primary_topic_field_id":   short_id(pfield.get("id")),
        "topics_field_ids": [
            short_id((t.get("field") or {}).get("id"))
            for t in topics
            if (t.get("field") or {}).get("id")
        ],
        "referenced_works":         [short_id(r) for r in refs if short_id(r)],
    }


# --- Optional pre-flight quota probe ----------------------------------------
def check_openalex_quota():
    """Remaining daily credits, or None if the value can't be read."""
    try:
        r = requests.get("https://api.openalex.org/rate-limit",
                         params={"api_key": pyalex.config.api_key}, timeout=15)
        r.raise_for_status()
        data = r.json()
    except Exception:
        return None

    for key in ("remaining", "credits_remaining", "daily_remaining"):
        if key in data:
            return data[key]
    rl = data.get("rate_limit") or data.get("rateLimit") or {}
    for key in ("remaining", "credits_remaining"):
        if key in rl:
            return rl[key]
    return None


# --- Field → domain reference table -----------------------------------------
def fetch_fields_taxonomy():
    r = requests.get(
        "https://api.openalex.org/fields",
        params={"per-page": 200, "api_key": pyalex.config.api_key},
        timeout=30,
    )
    r.raise_for_status()
    rows = []
    for f in r.json()["results"]:
        dom = f.get("domain") or {}
        rows.append({
            "field_id":    short_id(f.get("id")),
            "field_name":  f.get("display_name"),
            "domain_id":   short_id(dom.get("id")),
            "domain_name": dom.get("display_name"),
        })
    return pd.DataFrame(rows)


# --- Crawl ------------------------------------------------------------------
def crawl(df, doi_col="doi",
          checkpoint_fn=None, checkpoint_every=CHECKPOINT_EVERY):
    """Returns (works_df, failed, stopped, reason).

    Never raises on quota or user interrupt.
    If `checkpoint_fn(partial_works_df, n_works)` is provided, it is invoked
    every `checkpoint_every` successfully processed batches. The callback
    receives ONLY this run's accumulated works — the caller is responsible
    for merging with pre-existing state (save_works already does this).
    Callback exceptions are logged but do not abort the crawl.
    """
    dois = df[doi_col].map(normalize_doi).dropna().drop_duplicates().tolist()
    batches = [dois[i:i + BATCH_SIZE] for i in range(0, len(dois), BATCH_SIZE)]
    print(f"{len(dois)} unique DOIs → {len(batches)} batches")

    stop  = threading.Event()
    rows, failed = [], []
    stopped, reason = False, None

    futures = {}
    pool = ThreadPoolExecutor(max_workers=MAX_WORKERS)
    try:
        futures = {pool.submit(fetch_batch, b, stop): b for b in batches}

        pbar = tqdm(as_completed(futures), total=len(futures),
                    desc="Batches", unit="batch", mininterval=0.5)
        try:
            for i, fut in enumerate(pbar, 1):
                try:
                    results = fut.result()
                except CrawlHalted as e:
                    stopped, reason = True, (reason or str(e))
                    stop.set()
                    pbar.set_postfix_str("HALTED")
                    tqdm.write(f"  ⚠ HALTED — {reason}")
                    break
                except Exception as e:
                    failed.append((futures[fut], e))
                    tqdm.write(f"  ✗ batch {i}/{len(batches)} failed: {e}")
                    continue

                if results:
                    rows.extend(to_row(w) for w in results)
                pbar.set_postfix_str(f"works={len(rows)} failed={len(failed)}")

                # ---- Periodic checkpoint -----------------------------------
                if checkpoint_fn and i % checkpoint_every == 0:
                    try:
                        checkpoint_fn(pd.DataFrame(rows), len(rows))
                        tqdm.write(f"  💾 checkpoint at {i}/{len(batches)}  "
                                   f"({len(rows)} works)")
                    except Exception as ck_err:
                        tqdm.write(f"  ⚠ checkpoint failed at {i}: {ck_err}")

        except KeyboardInterrupt:
            stopped, reason = True, "user interrupt"
            stop.set()
            tqdm.write("  ⏹ interrupted — saving partial results")
    finally:
        stop.set()
        for f in futures:
            f.cancel()
        try:
            pool.shutdown(wait=True, cancel_futures=True)     # Python 3.9+
        except TypeError:
            pool.shutdown(wait=True)

    return pd.DataFrame(rows), failed, stopped, reason


# --- Persistence helpers ----------------------------------------------------
def load_existing_works(path):
    """Return (df, set-of-already-fetched-DOIs). Empty if file doesn't exist."""
    if not path.exists():
        print(f"No previous {path.name} — starting fresh")
        return pd.DataFrame(), set()

    try:
        existing = pd.read_parquet(path)
    except Exception as e:
        raise RuntimeError(
            f"Found {path} but could not read it ({e}). "
            f"Move or delete it to start fresh."
        ) from e

    missing_cols = set(REQUESTED) - set(existing.columns)
    if missing_cols:
        print(f"⚠  {path.name} is missing columns: {sorted(missing_cols)}")
        print("   Those will be NaN for pre-existing rows. "
              "Delete the file to re-fetch from scratch.")

    already = set(existing["doi"].dropna()) if "doi" in existing.columns else set()
    print(f"Resuming: {len(existing)} works already present "
          f"({len(already)} distinct DOIs)")
    return existing, already


def save_works(combined, path, n_new, failed, verbose=True):
    """Dedupe, reindex, normalize list cols, write parquet, optionally report."""
    if "openalex_id" in combined.columns:
        combined = combined.drop_duplicates(subset="openalex_id", keep="first")
    else:
        combined = combined.drop_duplicates(subset="doi", keep="first")

    out = combined.reindex(columns=REQUESTED).copy()
    for col in LIST_COLS:
        out[col] = out[col].apply(lambda x: x if isinstance(x, list) else [])

    out.to_parquet(path, index=False, engine="pyarrow")
    if verbose:
        print(f"Wrote {len(out)} works → {path.name} "
              f"(+{n_new} new this run, {len(failed)} batches failed)")
    return out


# --- Main -------------------------------------------------------------------
if __name__ == "__main__":
    if CHECK_QUOTA_FIRST:
        remaining = check_openalex_quota()
        if remaining is not None:
            print(f"OpenAlex credits remaining today: {remaining}")
            if float(remaining) <= 0:
                print("Quota already exhausted — nothing fetched. "
                      "Existing data is untouched.")
                raise SystemExit(0)

    df = pd.read_parquet(DATA / "cache/dois.parquet")
    df["doi_norm"] = df["doi"].map(normalize_doi)

    works_path = DATA / "works.parquet"
    existing, already = load_existing_works(works_path)

    todo_mask = df["doi_norm"].notna() & ~df["doi_norm"].isin(already)
    df_todo   = df.loc[todo_mask, ["doi"]].drop_duplicates()

    print(f"{len(df_todo)} new DOIs to fetch "
          f"({len(df) - len(df_todo)} input rows already covered)")

    # Checkpoint callback: merges this run's partial data with the snapshot
    # taken at startup. save_works is idempotent (dedupe on openalex_id), so
    # repeated calls are safe and progressively supersede each other.
    def checkpoint(partial_works, n_works):
        combined = pd.concat([existing, partial_works], ignore_index=True)
        save_works(combined, works_path, n_new=n_works,
                   failed=[], verbose=False)

    new_works = pd.DataFrame()
    failed, stopped, reason = [], False, None

    try:
        if len(df_todo):
            new_works, failed, stopped, reason = crawl(
                df_todo,
                checkpoint_fn=checkpoint,
                checkpoint_every=CHECKPOINT_EVERY,
            )
        else:
            print("Nothing new to fetch — skipping API calls")
    finally:
        # Final save: catches works fetched after the last checkpoint boundary
        # and writes the definitive summary (including the failed list).
        combined = pd.concat([existing, new_works], ignore_index=True)
        out = save_works(combined, works_path, len(new_works), failed)

        if failed:
            print(f"\n{len(failed)} batches gave up after retries — their DOIs "
                  f"are NOT in works.parquet and will be retried next run:")
            for d, err in failed[:5]:
                label = f"{len(d)} DOIs" if isinstance(d, list) else str(d)
                print(f"  {label}: {err}")
            if len(failed) > 5:
                print(f"  … and {len(failed) - 5} more batches")

        if stopped:
            kind = ("user interrupt" if reason == "user interrupt"
                    else "OpenAlex quota/rate limit")
            print(f"\n⚠ STOPPED EARLY — {kind}.")
            print(f"  reason : {reason}")
            print(f"  saved  : {len(out)} works in total")
            print(f"  pending: up to {len(df_todo) - len(new_works)} DOIs still "
                  f"un-fetched (they are NOT marked as done)")
            print("  → rerun this cell to resume exactly where it stopped.")

        # Reference table — skip when we know the budget is gone
        if stopped and reason != "user interrupt":
            print("Skipping fields.parquet refresh (quota exhausted).")
        else:
            try:
                fields_df = fetch_fields_taxonomy()
                fields_df.to_parquet(DATA / "fields.parquet",
                                     index=False, engine="pyarrow")
                print(f"Wrote {len(fields_df)} fields → fields.parquet")
            except Exception as e:
                print(f"⚠ could not refresh fields.parquet: {e}")

OpenAlex credits remaining today: 9816


FileNotFoundError: ../data/external/cache/dois.parquet

In [10]:
import pandas as pd
len(pd.read_parquet("../data/works.parquet"))

348417

In [1]:
import os
import threading
from pathlib import Path

import pandas as pd
import requests
import pyalex
from pyalex import Works
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

# --- Config -----------------------------------------------------------------
pyalex.config.email   = os.environ['OPENALEX_EMAIL']
pyalex.config.api_key = os.environ['OPENALEX_API_KEY']

pyalex.config.max_retries      = 0
pyalex.config.retry_http_codes = [429, 500, 503]

# Pedimos únicamente lo necesario para ambas tablas
FIELDS            = ["ids", "open_access", "authorships"]
BATCH_SIZE        = 50
PER_PAGE          = 50
MAX_WORKERS       = 5
MAX_RETRIES       = 5
CHECKPOINT_EVERY  = 50
LONG_WAIT_SECONDS = 120
CHECK_QUOTA_FIRST = True

DATA               = Path("../data")
OA_PATH            = DATA / "work_open_access.parquet"
AUTH_PATH          = DATA / "work_authorships.parquet"
OA_COLS            = ["work_openalex_id", "is_open_access", "open_access_status"]
AUTH_COLS          = ["work_openalex_id", "author_id",
                      "institution_id", "institution_country", "institution_code"]

# Clave natural de cada tabla para deduplicar
OA_DEDUP   = ["work_openalex_id"]
AUTH_DEDUP = ["work_openalex_id", "author_id", "institution_id"]


class CrawlHalted(RuntimeError):
    """Hard stop: today's API budget is spent (or the crawl was aborted)."""


# --- Helpers ----------------------------------------------------------------
def short_id(url):
    """'https://openalex.org/P4310319965' -> 'P4310319965'; None-safe."""
    if not url:
        return None
    return url.rstrip("/").rsplit("/", 1)[-1]


def _quota_reason(exc):
    """Return a human reason if `exc` means today's budget is spent, else None."""
    resp = getattr(exc, "response", None)
    if getattr(resp, "status_code", None) != 429:
        return None

    h = getattr(resp, "headers", {}) or {}
    remaining   = h.get("X-RateLimit-Remaining")
    retry_after = h.get("Retry-After")

    try:
        if remaining is not None and int(float(remaining)) <= 0:
            return f"daily budget spent (X-RateLimit-Remaining={remaining})"
    except (TypeError, ValueError):
        pass
    try:
        if retry_after is not None and float(retry_after) > LONG_WAIT_SECONDS:
            return f"server asks for a {retry_after}s wait (Retry-After)"
    except (TypeError, ValueError):
        pass
    return None


# --- Fetch (by openalex_id) -------------------------------------------------
def fetch_batch(openalex_ids, stop, attempt=0):
    """Fetch one batch of works by OpenAlex ID. Raises CrawlHalted on quota."""
    if stop.is_set():
        return []

    try:
        return (Works()
                .filter_or(openalex_id=openalex_ids)
                .select(FIELDS)
                .get(per_page=PER_PAGE))

    except Exception as e:
        status = getattr(getattr(e, "response", None), "status_code", None)

        if status == 429:
            reason = _quota_reason(e)
            if reason:
                stop.set()
                raise CrawlHalted(reason) from e
            if attempt >= MAX_RETRIES:
                stop.set()
                raise CrawlHalted(f"persistent 429 after {MAX_RETRIES} retries") from e
            if stop.wait(min(2 ** attempt, 30)):
                raise CrawlHalted("stop requested during backoff") from e
            return fetch_batch(openalex_ids, stop, attempt + 1)

        # Non-429 4xx (bad ID / forbidden / not found) — retrying won't help.
        if status is not None and 400 <= status < 500:
            print(f"  ! HTTP {status} on batch of {len(openalex_ids)} ids — skipping (no retry)")
            return []

        # 5xx / network — retry, then give up on this batch.
        if attempt >= MAX_RETRIES:
            raise
        if stop.wait(min(2 ** attempt, 30)):
            raise CrawlHalted("stop requested during backoff") from e
        return fetch_batch(openalex_ids, stop, attempt + 1)


def to_oa_row(work):
    """Devuelve una fila para la tabla de open access."""
    ids = work.get("ids") or {}
    oa  = work.get("open_access") or {}
    return {
        "work_openalex_id":   short_id(ids.get("openalex")),
        "is_open_access":     oa.get("is_oa"),
        "open_access_status": oa.get("oa_status"),
    }


def to_auth_rows(work):
    """Devuelve una lista de filas, una por cada institución de cada autoría."""
    ids = work.get("ids") or {}
    work_id = short_id(ids.get("openalex"))
    rows = []
    for authorship in work.get("authorships") or []:
        author_id = short_id((authorship.get("author") or {}).get("id"))
        for inst in authorship.get("institutions") or []:
            rows.append({
                "work_openalex_id":    work_id,
                "author_id":           author_id,
                "institution_id":      short_id(inst.get("id")),
                "institution_country": inst.get("country_code"),
                "institution_code":    inst.get("ror"),   # ROR como "código"
            })
    return rows


# --- Optional pre-flight quota probe ----------------------------------------
def check_openalex_quota():
    """Remaining daily credits, or None if the value can't be read."""
    try:
        r = requests.get("https://api.openalex.org/rate-limit",
                         params={"api_key": pyalex.config.api_key}, timeout=15)
        r.raise_for_status()
        data = r.json()
    except Exception:
        return None

    for key in ("remaining", "credits_remaining", "daily_remaining"):
        if key in data:
            return data[key]
    rl = data.get("rate_limit") or data.get("rateLimit") or {}
    for key in ("remaining", "credits_remaining"):
        if key in rl:
            return rl[key]
    return None


# --- Crawl ------------------------------------------------------------------
def crawl(openalex_ids, checkpoint_fn=None, checkpoint_every=CHECKPOINT_EVERY):
    """Devuelve (oa_df, auth_df, failed, stopped, reason)."""
    batches = [openalex_ids[i:i + BATCH_SIZE]
               for i in range(0, len(openalex_ids), BATCH_SIZE)]
    print(f"{len(openalex_ids)} unique OpenAlex IDs → {len(batches)} batches")

    stop  = threading.Event()
    oa_rows, auth_rows, failed = [], [], []
    stopped, reason = False, None

    futures = {}
    pool = ThreadPoolExecutor(max_workers=MAX_WORKERS)
    try:
        futures = {pool.submit(fetch_batch, b, stop): b for b in batches}

        pbar = tqdm(as_completed(futures), total=len(futures),
                    desc="Batches", unit="batch", mininterval=0.5)
        try:
            for i, fut in enumerate(pbar, 1):
                try:
                    results = fut.result()
                except CrawlHalted as e:
                    stopped, reason = True, (reason or str(e))
                    stop.set()
                    pbar.set_postfix_str("HALTED")
                    tqdm.write(f"  ⚠ HALTED — {reason}")
                    break
                except Exception as e:
                    failed.append((futures[fut], e))
                    tqdm.write(f"  ✗ batch {i}/{len(batches)} failed: {e}")
                    continue

                if results:
                    for work in results:
                        oa_rows.append(to_oa_row(work))
                        auth_rows.extend(to_auth_rows(work))

                pbar.set_postfix_str(
                    f"oa={len(oa_rows)} auth={len(auth_rows)} failed={len(failed)}"
                )

                # ---- Checkpoint periódico ----------------------------------
                if checkpoint_fn and i % checkpoint_every == 0:
                    try:
                        checkpoint_fn(pd.DataFrame(oa_rows),
                                      pd.DataFrame(auth_rows),
                                      len(oa_rows), len(auth_rows))
                        tqdm.write(
                            f"  💾 checkpoint at {i}/{len(batches)}  "
                            f"({len(oa_rows)} OA rows, {len(auth_rows)} auth rows)"
                        )
                    except Exception as ck_err:
                        tqdm.write(f"  ⚠ checkpoint failed at {i}: {ck_err}")

        except KeyboardInterrupt:
            stopped, reason = True, "user interrupt"
            stop.set()
            tqdm.write("  ⏹ interrupted — saving partial results")
    finally:
        stop.set()
        for f in futures:
            f.cancel()
        try:
            pool.shutdown(wait=True, cancel_futures=True)
        except TypeError:
            pool.shutdown(wait=True)

    return (pd.DataFrame(oa_rows), pd.DataFrame(auth_rows),
            failed, stopped, reason)


# --- Persistence ------------------------------------------------------------
def load_existing(path, id_col):
    """Devuelve (df, set de IDs ya presentes). Vacío si el archivo no existe."""
    if not path.exists():
        print(f"No previous {path.name} — starting fresh")
        return pd.DataFrame(), set()

    try:
        existing = pd.read_parquet(path)
    except Exception as e:
        raise RuntimeError(
            f"Found {path} but could not read it ({e}). "
            f"Move or delete it to start fresh."
        ) from e

    already = (set(existing[id_col].dropna())
               if id_col in existing.columns else set())
    print(f"Resuming {path.name}: {len(existing)} rows "
          f"({len(already)} distinct {id_col})")
    return existing, already


def save(combined, path, cols, dedup_on, n_new, failed, verbose=True):
    """Dedupe por `dedup_on`, reindex, escribe parquet y reporta."""
    combined = combined.drop_duplicates(subset=dedup_on, keep="first")
    out = combined.reindex(columns=cols).copy()
    out.to_parquet(path, index=False, engine="pyarrow")
    if verbose:
        print(f"Wrote {len(out)} rows → {path.name} "
              f"(+{n_new} new this run, {len(failed)} batches failed)")
    return out


# --- Main -------------------------------------------------------------------
if __name__ == "__main__":
    if CHECK_QUOTA_FIRST:
        remaining = check_openalex_quota()
        if remaining is not None:
            print(f"OpenAlex credits remaining today: {remaining}")
            if float(remaining) <= 0:
                print("Quota already exhausted — nothing fetched. "
                      "Existing data is untouched.")
                raise SystemExit(0)

    works_path = DATA / "works.parquet"
    if not works_path.exists():
        raise SystemExit(f"{works_path} not found — run the main crawl first.")

    works = pd.read_parquet(works_path, columns=["openalex_id"])
    all_ids = works["openalex_id"].dropna().drop_duplicates().tolist()
    print(f"{len(all_ids)} unique OpenAlex IDs in {works_path.name}")

    # Cargamos el estado previo de ambas tablas por separado
    oa_existing, oa_done     = load_existing(OA_PATH,   "work_openalex_id")
    auth_existing, auth_done = load_existing(AUTH_PATH, "work_openalex_id")

    # Un ID se considera "hecho" si aparece en cualquiera de las dos tablas:
    # pedimos ambos campos a la vez, así que cualquiera de ellas confirma
    # que el trabajo ya se consultó.
    already_done = oa_done | auth_done
    todo_ids = [i for i in all_ids if i not in already_done]
    print(f"{len(todo_ids)} IDs still need backfill")

    # Checkpoint: fusiona los parciales con los snapshots previos y guarda
    def checkpoint(oa_partial, auth_partial, n_oa, n_auth):
        oa_comb   = pd.concat([oa_existing, oa_partial], ignore_index=True)
        auth_comb = pd.concat([auth_existing, auth_partial], ignore_index=True)
        save(oa_comb,   OA_PATH,   OA_COLS,   OA_DEDUP,
             n_oa,   failed=[], verbose=False)
        save(auth_comb, AUTH_PATH, AUTH_COLS, AUTH_DEDUP,
             n_auth, failed=[], verbose=False)

    new_oa = pd.DataFrame()
    new_auth = pd.DataFrame()
    failed, stopped, reason = [], False, None

    try:
        if todo_ids:
            new_oa, new_auth, failed, stopped, reason = crawl(
                todo_ids,
                checkpoint_fn=checkpoint,
                checkpoint_every=CHECKPOINT_EVERY,
            )
        else:
            print("Nothing to backfill — skipping API calls")
    finally:
        # Guardado final (captura lo que quedó después del último checkpoint)
        oa_combined   = pd.concat([oa_existing, new_oa], ignore_index=True)
        auth_combined = pd.concat([auth_existing, new_auth], ignore_index=True)
        out_oa   = save(oa_combined,   OA_PATH,   OA_COLS,   OA_DEDUP,
                        len(new_oa),   failed)
        out_auth = save(auth_combined, AUTH_PATH, AUTH_COLS, AUTH_DEDUP,
                        len(new_auth), failed)

        if failed:
            print(f"\n{len(failed)} batches gave up after retries — their IDs "
                  f"are NOT in the output files and will be retried next run:")
            for ids, err in failed[:5]:
                label = f"{len(ids)} ids" if isinstance(ids, list) else str(ids)
                print(f"  {label}: {err}")
            if len(failed) > 5:
                print(f"  … and {len(failed) - 5} more batches")

        if stopped:
            kind = ("user interrupt" if reason == "user interrupt"
                    else "OpenAlex quota/rate limit")
            print(f"\n⚠ STOPPED EARLY — {kind}.")
            print(f"  reason : {reason}")
            print(f"  saved  : {len(out_oa)} OA rows, {len(out_auth)} authorship rows")
            print(f"  pending: up to {len(todo_ids) - len(new_oa)} IDs still "
                  f"un-fetched (they are NOT marked as done)")
            print("  → rerun to resume exactly where it stopped.")

OpenAlex credits remaining today: 4549
348417 unique OpenAlex IDs in works.parquet
Resuming work_open_access.parquet: 259999 rows (259999 distinct work_openalex_id)
Resuming work_authorships.parquet: 3760725 rows (248809 distinct work_openalex_id)
88418 IDs still need backfill
88418 unique OpenAlex IDs → 1769 batches


Batches:   3%|▎         | 51/1769 [00:08<07:01,  4.08batch/s, oa=2999 auth=22901 failed=0]

  💾 checkpoint at 50/1769  (2499 OA rows, 18773 auth rows)


Batches:   6%|▌         | 100/1769 [00:17<08:20,  3.34batch/s, oa=5499 auth=52417 failed=0]

  💾 checkpoint at 100/1769  (4999 OA rows, 45863 auth rows)


Batches:   9%|▊         | 151/1769 [00:25<06:35,  4.09batch/s, oa=8049 auth=85124 failed=0]

... (63 outputs left out)

Batches:  99%|█████████▉| 1751/1769 [07:57<00:08,  2.21batch/s, oa=88249 auth=1229148 failed=0]

  💾 checkpoint at 1750/1769  (87499 OA rows, 1217485 auth rows)


Batches: 100%|██████████| 1769/1769 [07:58<00:00,  3.69batch/s, oa=88417 auth=1231986 failed=0]


Wrote 348416 rows → work_open_access.parquet (+88417 new this run, 0 batches failed)
Wrote 4980147 rows → work_authorships.parquet (+1231986 new this run, 0 batches failed)
